# Day 24 Tutorial：混合模型真实消融

> **课程附带教程，不是学习者实验记录。** 所有模型只预测 ESOL logS，不支持粘合剂结论。

## Goal

在同一 ESOL scaffold 外部验证和同一 GroupKFold split 上，比较 `tree_only`、`mlp_only`、`simple_mean`、`stack_tree_only`、`stack_tree_mlp`，从而区分“树本身”和“保留二层但移除 MLP”。


## Setup

所有变体运行前固定。树模型读取 Day 07 随机森林配置，MLP 与 Day 21–23 完全相同。两个 stack 共享同一 scaffold-aware split 列表和 Ridge 二层。


In [ ]:
from pathlib import Path
import contextlib
import io

from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import deepchem as dc

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "public" / "esol.md").exists():
            return candidate
    raise RuntimeError("请从 ML-practice 仓库内运行本教程。")

REPO_ROOT = find_repo_root()
CACHE_DIR = REPO_ROOT / ".cache" / "deepchem"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tasks, datasets, transformers = dc.molnet.load_delaney(
        featurizer=featurizer,
        splitter="scaffold",
        transformers=[],
        reload=True,
        data_dir=str(CACHE_DIR),
        save_dir=str(CACHE_DIR),
    )

train_dataset, valid_dataset, _sealed_test_dataset = datasets
X_train = np.asarray(train_dataset.X)
y_train = np.asarray(train_dataset.y).reshape(-1)
X_valid = np.asarray(valid_dataset.X)
y_valid = np.asarray(valid_dataset.y).reshape(-1)
train_ids = np.asarray(train_dataset.ids).astype(str)
valid_ids = np.asarray(valid_dataset.ids).astype(str)

assert transformers == []
assert X_train.shape == (902, 1024) and y_train.shape == (902,)
assert X_valid.shape == (113, 1024) and y_valid.shape == (113,)
print("Task:", tasks[0])
print("Train / valid:", X_train.shape, X_valid.shape)
print("测试集对象保持封存，本教程不创建测试预测。")


In [ ]:
import json
import warnings
from time import perf_counter

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def murcko_group(smiles):
    molecule = Chem.MolFromSmiles(str(smiles))
    if molecule is None:
        raise ValueError(f"无法解析 SMILES：{smiles}")
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=molecule, includeChirality=False
    )
    return scaffold or "__ACYCLIC__"

train_scaffolds = np.asarray([murcko_group(item) for item in train_ids])
valid_scaffolds = np.asarray([murcko_group(item) for item in valid_ids])
assert set(train_scaffolds).isdisjoint(set(valid_scaffolds))

inner_splits = list(
    GroupKFold(n_splits=5).split(
        X_train, y_train, groups=train_scaffolds
    )
)
coverage = np.zeros(len(y_train), dtype=int)
for fit_idx, hold_idx in inner_splits:
    assert set(train_scaffolds[fit_idx]).isdisjoint(
        set(train_scaffolds[hold_idx])
    )
    coverage[hold_idx] += 1
assert np.all(coverage == 1)

config_path = (
    REPO_ROOT
    / "experiments"
    / "esol"
    / "day01_baseline"
    / "results"
    / "run_config.json"
)
if not config_path.exists():
    raise FileNotFoundError(
        "缺少 Day 07 冻结配置，无法复现树模型：" + str(config_path)
    )
frozen = json.loads(config_path.read_text(encoding="utf-8"))
rf_params = frozen["model_params"]["random_forest"]

def make_tree():
    return RandomForestRegressor(**rf_params)

def make_mlp():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(32,),
            alpha=0.001,
            early_stopping=True,
            max_iter=300,
            n_iter_no_change=10,
            random_state=SEED,
        ),
    )

print(
    "Train / valid scaffold groups:",
    len(np.unique(train_scaffolds)),
    len(np.unique(valid_scaffolds)),
)
print("GroupKFold holdout sizes:", [len(hold) for _, hold in inner_splits])


## Steps

### 1. 创建真正不同的预注册变体

`stack_tree_only` 仍然从 OOF 树预测训练 Ridge 二层；因此它不是 `tree_only` 预测的重复别名。


In [ ]:
tree = make_tree()
mlp = make_mlp()
stack_tree_only = StackingRegressor(
    estimators=[("tree", make_tree())],
    final_estimator=Ridge(alpha=1.0),
    cv=inner_splits,
    passthrough=False,
    n_jobs=1,
)
stack_tree_mlp = StackingRegressor(
    estimators=[
        ("tree", make_tree()),
        ("mlp", make_mlp()),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=inner_splits,
    passthrough=False,
    n_jobs=1,
)
preregistered_variants = [
    "tree_only",
    "mlp_only",
    "simple_mean",
    "stack_tree_only",
    "stack_tree_mlp",
]
print("Pre-registered:", preregistered_variants)


### 2. 产生所有预注册预测并记录成本

简单平均复用两个单模型，成本记为两次单模型拟合之和。收敛警告只捕获和报告，不隐藏。


In [ ]:
predictions = {}
costs = {}
warning_rows = []

started = perf_counter()
fitted_tree = clone(tree).fit(X_train, y_train)
costs["tree_only"] = perf_counter() - started
predictions["tree_only"] = fitted_tree.predict(X_valid)

started = perf_counter()
with warnings.catch_warnings(record=True) as mlp_caught:
    warnings.simplefilter("always", ConvergenceWarning)
    fitted_mlp = clone(mlp).fit(X_train, y_train)
costs["mlp_only"] = perf_counter() - started
predictions["mlp_only"] = fitted_mlp.predict(X_valid)

predictions["simple_mean"] = (
    predictions["tree_only"] + predictions["mlp_only"]
) / 2
costs["simple_mean"] = costs["tree_only"] + costs["mlp_only"]

started = perf_counter()
fitted_stack_tree_only = stack_tree_only.fit(X_train, y_train)
costs["stack_tree_only"] = perf_counter() - started
predictions["stack_tree_only"] = (
    fitted_stack_tree_only.predict(X_valid)
)

started = perf_counter()
with warnings.catch_warnings(record=True) as full_stack_caught:
    warnings.simplefilter("always", ConvergenceWarning)
    fitted_stack_tree_mlp = stack_tree_mlp.fit(X_train, y_train)
costs["stack_tree_mlp"] = perf_counter() - started
predictions["stack_tree_mlp"] = (
    fitted_stack_tree_mlp.predict(X_valid)
)

for component, caught in [
    ("mlp_only", mlp_caught),
    ("stack_tree_mlp", full_stack_caught),
]:
    messages = [
        str(item.message)
        for item in caught
        if issubclass(item.category, ConvergenceWarning)
    ]
    warning_rows.append({
        "component": component,
        "convergence_warning_count": len(messages),
        "messages": " | ".join(messages[:3]) or "none",
    })
warning_report = pd.DataFrame(warning_rows)
display(warning_report)


### 3. 统一计算消融指标与差值

`delta_vs_tree < 0` 只表示当前固定验证 RMSE 更低。`stack_tree_only` 和 `tree_only` 是不同拟合机制，即使偶然产生相近数值也不能合并成一行。


In [ ]:
baseline_rmse = root_mean_squared_error(
    y_valid, predictions["tree_only"]
)
rows = []
for name in preregistered_variants:
    prediction = predictions[name]
    rmse = root_mean_squared_error(y_valid, prediction)
    rows.append({
        "variant": name,
        "rmse": rmse,
        "delta_vs_tree": rmse - baseline_rmse,
        "mae": mean_absolute_error(y_valid, prediction),
        "r2": r2_score(y_valid, prediction),
        "fit_seconds_teaching_run": costs[name],
    })
ablation = pd.DataFrame(rows)
display(ablation.round(4))


## Checks

结果必须完整保留预注册顺序；每个变体使用相同验证标签、模型配方和 scaffold splits。特别检查两个 stack 的基础模型数量不同。


In [ ]:
assert ablation["variant"].tolist() == preregistered_variants
assert set(predictions) == set(preregistered_variants)
assert all(len(pred) == len(y_valid) for pred in predictions.values())
assert np.isfinite(ablation[[
    "rmse", "mae", "r2", "fit_seconds_teaching_run"
]]).all().all()
assert np.isclose(
    ablation.loc[
        ablation["variant"] == "tree_only", "delta_vs_tree"
    ].iloc[0],
    0.0,
)
assert len(stack_tree_only.estimators) == 1
assert len(stack_tree_mlp.estimators) == 2
assert not np.allclose(
    predictions["tree_only"], predictions["stack_tree_only"]
)
assert stack_tree_only.cv is inner_splits
assert stack_tree_mlp.cv is inner_splits
assert set(train_scaffolds).isdisjoint(set(valid_scaffolds))
for fit_idx, hold_idx in inner_splits:
    assert set(train_scaffolds[fit_idx]).isdisjoint(
        set(train_scaffolds[hold_idx])
    )
print("Real-ablation and scaffold-boundary checks passed.")


## Next Steps

在个人实验中对预先声明的多个种子重复整个矩阵，报告均值、标准差和逐种子差值。没有稳定收益时，保留单模型或简单平均也是完整结论。
